[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/entregables/01_eda_hallazgos.ipynb)

# Análisis Exploratorio — Detección de Deslizamientos con Machine Learning

**Curso:** Visualización de Datos  
**Dataset:** Landslide4Sense — imágenes satelitales multiespectrales (14 bandas)  
**Pregunta central:** ¿Qué aprendemos de los datos antes de asumir cualquier hipótesis sobre los modelos?

---
Este notebook presenta cinco hallazgos exploratorios. No hay conclusiones anticipadas — solo observaciones.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D

IN_COLAB = 'google.colab' in __import__('sys').modules
if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)

# Estilo global — data-ink mínimo
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#CCCCCC', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#EEEEEE', 'grid.linewidth': 0.8,
    'font.family': 'sans-serif', 'font.size': 11,
    'xtick.color': '#555555', 'ytick.color': '#555555', 'axes.labelcolor': '#333333',
})

# Paleta por grupo de sensor
COLORES = {
    'RedEdge': '#DC2626',
    'Topo':    '#F97316',
    'SAR':     '#EAB308',
    'Optico':  '#3B82F6',
    'enfasis': '#DC2626',
    'gris':    '#9CA3AF',
    'gris_claro': '#E5E7EB',
}

# Cargar datos
df = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
for col in ['F1 medio','Std','AUC-ROC','Precisión','Recall','IoU']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

ch = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')

# Cargar folds
FOLDS_FALLBACK = {
    'LR':  [0.7929, 0.7681, 0.8232, 0.7772, 0.7813],
    'SVM': [0.7526, 0.7731, 0.8100, 0.8325, 0.8189],
    'RF':  [0.8363, 0.8238, 0.8480, 0.8401, 0.8361],
    'ResNet-50':    [0.7762, 0.7708, 0.8154, 0.7636, 0.8065],
    'U-Net ResNet34': [0.6855, 0.7084, 0.7061, 0.6900, 0.6842],
}
try:
    def load_f(fname, key='best_f1', alt_key='f1_pixel_thr05'):
        with open(f'{DATA_DIR}/folds/{fname}') as f:
            d = json.load(f)
        return [x.get(key, x.get(alt_key, 0)) for x in d['folds']]
    fold_data = {
        'LR':  load_f('logistic_regression_folds.json'),
        'SVM': load_f('svm_folds.json'),
        'RF':  load_f('random_forest_folds.json'),
        'ResNet-50':     load_f('resnet50_folds.json', key='f1_thr05'),
        'U-Net ResNet34': load_f('unet_folds.json', key='f1_pixel_thr05'),
    }
except FileNotFoundError:
    fold_data = FOLDS_FALLBACK.copy()

print('Datos cargados correctamente.')

---
## Hallazgo 1 — Rendimiento global por modelo
**Pregunta:** ¿Qué tan bien detecta cada modelo un deslizamiento de tierra?

Métrica principal: **F1-Score** (equilibrio entre detectar los que sí ocurren y no generar falsas alarmas).

In [ ]:
modelos = ['LR', 'SVM', 'RF', 'ResNet-50', 'EfficientNet', 'U-Net']
f1_vals = [0.7886, 0.7974, 0.8368, 0.7840, 0.7554, 0.4443]
tipos   = ['Clásico','Clásico','Clásico','DL','DL','DL']
colores = ['#6B7280' if t == 'Clásico' else '#9CA3AF' for t in tipos]
# RF y el punto de referencia se destacan
colores[2] = COLORES['enfasis']   # RF — mejor modelo clásico

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(modelos, f1_vals, color=colores, height=0.55, zorder=3)

# Línea de referencia F1 = 0.80
ax.axvline(0.80, color='#374151', lw=1.4, ls='--', zorder=2)
ax.text(0.801, 5.6, 'Umbral F1 = 0.80', color='#374151', fontsize=9, va='top')

# Valores al final de cada barra
for bar, val in zip(bars, f1_vals):
    ax.text(val + 0.004, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='#374151')

# Anotación narrativa
ax.annotate('Random Forest supera\nel umbral de referencia',
            xy=(0.8368, 2), xytext=(0.72, 1.2),
            arrowprops=dict(arrowstyle='->', color=COLORES['enfasis'], lw=1.5),
            fontsize=9, color=COLORES['enfasis'])

ax.annotate('U-Net — el modelo\nmás complejo — queda\n43 puntos por debajo',
            xy=(0.4443, 5), xytext=(0.55, 4.6),
            arrowprops=dict(arrowstyle='->', color='#374151', lw=1.2),
            fontsize=9, color='#374151')

ax.set_xlim(0.0, 0.93)
ax.set_xlabel('F1-Score (0 = no detecta nada · 1 = perfecto)')
ax.set_title('F1-Score por Modelo — Detección de Deslizamientos', fontsize=13, pad=12)

# Leyenda mínima
leyenda = [
    mpatches.Patch(color='#6B7280', label='Modelos clásicos (Machine Learning tradicional)'),
    mpatches.Patch(color='#9CA3AF', label='Deep Learning (redes neuronales)'),
    mpatches.Patch(color=COLORES['enfasis'], label='Modelo con mejor resultado'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9, frameon=False)

ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_h1_f1_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Hallazgo 2 — Precisión vs Cobertura
**Pregunta:** ¿Los modelos detectan muchos deslizamientos (alta cobertura) o son muy precisos al hacerlo?

En detección de desastres, **fallar en detectar uno real (baja cobertura)** suele ser más costoso que una falsa alarma (baja precisión). ¿Dónde cae cada modelo?

In [ ]:
datos_pr = {
    'LR':        (0.7971, 0.7806),
    'SVM':       (0.8193, 0.7777),
    'RF':        (0.7439, 0.9569),
    'ResNet-50': (0.7219, 0.8771),
}
colores_pr = {
    'LR':'#6B7280', 'SVM':'#6B7280',
    'RF': COLORES['enfasis'], 'ResNet-50':'#374151',
}

# Literatura: (etiqueta, F1, color)
literatura = [
    ('Ghorbanzadeh et al. (2022)',        0.717, '#FECACA'),
    ('Lv et al. — L4S Competition (2022)',0.739, '#F87171'),
    ('Liu et al. — Multi-scale (2024)',   0.760, '#EF4444'),
    ('Enhanced U-Net++ (2025)',           0.841, '#B91C1C'),
]

fig, ax = plt.subplots(figsize=(7, 6.5))
r_arr = np.linspace(0.62, 0.999, 400)

for nombre, f1_val, cl in literatura:
    p_arr = f1_val * r_arr / (2 * r_arr - f1_val)
    mask = (p_arr > 0) & (p_arr <= 1.0)
    ax.plot(r_arr[mask], p_arr[mask], color=cl, lw=1.0, ls='--', zorder=1)
    # etiqueta en el extremo derecho de cada curva
    valid_r = r_arr[mask]
    valid_p = p_arr[mask]
    if len(valid_r):
        ax.text(valid_r[-1] - 0.005, valid_p[-1] + 0.010, nombre,
                fontsize=7.5, color=cl, ha='right', va='bottom')

for nm, (prec, rec) in datos_pr.items():
    ax.scatter(rec, prec, s=110, color=colores_pr[nm],
               zorder=5, edgecolors='white', linewidth=1.5)
    offset = {'LR':(-0.012, 0.013), 'SVM':(0.005, 0.013),
              'RF':(0.005, -0.022), 'ResNet-50':(0.005, 0.013)}
    dx, dy = offset.get(nm, (0.005, 0.013))
    ax.text(rec+dx, prec+dy, nm, fontsize=9,
            color=colores_pr[nm], fontweight='bold')

ax.annotate('RF: detecta el 95% de eventos\n(mayor cobertura)',
            xy=(0.9569, 0.7439), xytext=(0.785, 0.865),
            arrowprops=dict(arrowstyle='->', color=COLORES['enfasis'], lw=1.4),
            fontsize=8.5, color=COLORES['enfasis'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=COLORES['enfasis'], alpha=0.92))

ax.set_xlim(0.62, 1.02)
ax.set_ylim(0.62, 0.90)
ax.set_xlabel('Cobertura (Recall) — fracción de deslizamientos reales detectados')
ax.set_ylabel('Precisión — de las alertas generadas, ¿cuántas son reales?')
ax.set_title('Precisión vs Cobertura por Modelo', fontsize=13, pad=10)

ax.plot([], [], color='#EF4444', ls='--', lw=1, label='Benchmarks de literatura (curvas ISO-F1)')
ax.legend(fontsize=8.5, frameon=False, loc='upper left')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_h2_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Hallazgo 3 — ¿Qué información satelital importa más?
**Pregunta:** De las 14 bandas espectrales disponibles, ¿cuáles separan mejor los pixeles con deslizamiento de los que no lo tienen?

Se mide la **brecha (Δ)** entre el valor medio de reflectancia en zonas con y sin deslizamiento.

In [ ]:
canales = [
    ('S2-B7 RedEdge3', 0.8073, 'RedEdge'),
    ('S2-B6 RedEdge2', 0.5625, 'RedEdge'),
    ('ALOS DEM',       0.1954, 'Topo'),
    ('S1-VH SAR',      0.1882, 'SAR'),
    ('DEM Slope',      0.0430, 'Topo'),
    ('S2-B8A NIR-A',   0.0221, 'Optico'),
]
nombres = [c[0] for c in canales]
deltas  = [c[1] for c in canales]
grupos  = [c[2] for c in canales]
cols    = [COLORES[g] for g in grupos]

UMBRAL = 0.12   # barras menores a este umbral → etiqueta fuera

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(nombres, deltas, color=cols, height=0.55, zorder=3)

for bar, val, grp in zip(bars, deltas, grupos):
    bw = bar.get_width()
    y_mid = bar.get_y() + bar.get_height() / 2
    if bw >= UMBRAL:
        # valor al final de la barra (normal)
        ax.text(bw + 0.008, y_mid, f'{val:.3f}', va='center',
                fontsize=10, color='#374151')
    else:
        # barra pequeña: valor con caja de fondo para que resalte
        ax.text(bw + 0.018, y_mid, f'{val:.3f}', va='center',
                fontsize=9.5, color='#374151',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#F9FAFB',
                          edgecolor=COLORES[grp], alpha=0.9, linewidth=0.8))

ax.annotate('RedEdge3 tiene una brecha 4x\nmayor que SAR-VH',
            xy=(0.8073, 0), xytext=(0.5, 0.8),
            arrowprops=dict(arrowstyle='->', color=COLORES['enfasis'], lw=1.4),
            fontsize=9, color=COLORES['enfasis'])

ax.set_xlim(0, 0.95)
ax.set_xlabel('Brecha de señal (Δ) entre zonas con y sin deslizamiento')
ax.set_ylabel('Canal satelital')
ax.set_title('Canales Satelitales más Discriminativos', fontsize=13, pad=10)

leyenda = [
    mpatches.Patch(color=COLORES['RedEdge'], label='Banda RedEdge (Sentinel-2)'),
    mpatches.Patch(color=COLORES['Topo'],    label='Topografía (DEM/Pendiente)'),
    mpatches.Patch(color=COLORES['SAR'],     label='Radar SAR (Sentinel-1)'),
    mpatches.Patch(color=COLORES['Optico'],  label='Óptico NIR (Sentinel-2)'),
]
ax.legend(handles=leyenda, loc='upper right', fontsize=9,
          frameon=True, framealpha=0.95, edgecolor='#E5E7EB')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_h3_canales.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Hallazgo 4 — Brecha de señal entre clases
**Pregunta:** ¿Cómo se diferencian concretamente los valores de reflectancia entre pixeles con y sin deslizamiento en los canales más importantes?

Dot plot: cada punto sólido es el valor medio en zonas **con deslizamiento**, el hueco es sin deslizamiento. La distancia entre ellos es la brecha Δ.

In [ ]:
# Top 6 canales
dot_data = [
    ('S2-B7 RedEdge3', 2.0209, 1.2136, 'RedEdge'),
    ('S2-B6 RedEdge2', 1.4782, 0.9157, 'RedEdge'),
    ('ALOS DEM',       1.2739, 1.0786, 'Topo'),
    ('S1-VH SAR',      1.2488, 1.0606, 'SAR'),
    ('S2-B8A NIR-A',   1.0397, 1.0176, 'Optico'),
    ('DEM Slope',      1.0703, 1.0274, 'Topo'),
]

nombres_d = [d[0] for d in dot_data]
pos_vals  = [d[1] for d in dot_data]
neg_vals  = [d[2] for d in dot_data]
grupos_d  = [d[3] for d in dot_data]
cols_d    = [COLORES[g] for g in grupos_d]

fig, ax = plt.subplots(figsize=(9, 4.5))
y_pos = range(len(dot_data))

for i, (nm, pos, neg, grp) in enumerate(dot_data):
    col = COLORES[grp]
    ax.plot([neg, pos], [i, i], color='#D1D5DB', lw=1.8, zorder=2)
    ax.scatter(pos, i, s=90, color=col, zorder=4, marker='o')
    ax.scatter(neg, i, s=90, color='white', zorder=4, marker='o',
               edgecolors=col, linewidths=1.8)
    delta = pos - neg
    ax.text(max(pos, neg) + 0.04, i, f'Δ={delta:.2f}',
            va='center', fontsize=9, color='#6B7280')

ax.set_yticks(list(y_pos))
ax.set_yticklabels(nombres_d)
ax.set_xlim(0.8, 2.5)
ax.set_xlabel('Reflectancia media normalizada')
titulo = 'Brecha de Señal entre Clases — Top 6 Canales\n(sólido = con deslizamiento  ·  hueco = sin deslizamiento  ·  Δ = brecha)'
ax.set_title(titulo, fontsize=11, pad=10)

leyenda = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#555', markersize=9, label='Con deslizamiento'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor='#555', markeredgewidth=1.5, markersize=9, label='Sin deslizamiento'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9, frameon=False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_h4_dot_clases.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Hallazgo 5 — Variabilidad entre experimentos (5 folds)
**Pregunta:** ¿El rendimiento de cada modelo es estable o depende del subconjunto de datos que se use?

Un modelo inconsistente entre folds es señal de que puede fallar en condiciones no vistas. Cada punto es el resultado en uno de los 5 subconjuntos de prueba.

In [ ]:
modelos_f = list(fold_data.keys())
valores_f = list(fold_data.values())
medias_f  = [np.mean(v) for v in valores_f]

# Ordenar por media descendente
orden = np.argsort(medias_f)[::-1]
modelos_ord = [modelos_f[i] for i in orden]
valores_ord = [valores_f[i] for i in orden]
medias_ord  = [medias_f[i] for i in orden]

# Color: RF en rojo, resto gris
col_box = [COLORES['enfasis'] if 'RF' in m else '#9CA3AF' for m in modelos_ord]

fig, ax = plt.subplots(figsize=(9, 5))

bp = ax.boxplot(valores_ord, vert=False, patch_artist=True,
                widths=0.45, showfliers=False,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color='#9CA3AF'),
                capprops=dict(color='#9CA3AF'),
                boxprops=dict(linewidth=0))

for patch, col in zip(bp['boxes'], col_box):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)

# Puntos individuales (5 folds)
for i, (vals, col) in enumerate(zip(valores_ord, col_box), start=1):
    jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(vals))
    ax.scatter(vals, [i + j for j in jitter], color=col, s=45, zorder=5, alpha=0.9)

# Etiquetas de media
for i, (med, col) in enumerate(zip(medias_ord, col_box), start=1):
    ax.text(med + 0.003, i + 0.28, f'x̄={med:.3f}',
            fontsize=8.5, color=col, va='bottom')

# Anotación
ax.axvline(0.80, color='#374151', lw=1.2, ls='--', zorder=2)
ax.text(0.801, 0.55, 'F1 = 0.80', fontsize=8.5, color='#374151')

ax.set_yticks(range(1, len(modelos_ord)+1))
ax.set_yticklabels(modelos_ord)
ax.set_xlabel('F1-Score por fold (cada punto = un experimento)')
ax.set_title('Consistencia entre Experimentos — 5 Folds por Modelo', fontsize=13, pad=10)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_h5_folds.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Síntesis de los 5 hallazgos

| # | Hallazgo | Observación |
|---|----------|-------------|
| 1 | **F1 por modelo** | Random Forest supera el umbral F1=0.80; U-Net queda muy por debajo |
| 2 | **Precisión vs Cobertura** | RF prioriza cobertura (Recall=0.96); SVM prioriza precisión |
| 3 | **Canales satelitales** | Las bandas RedEdge dominan la discriminación entre clases |
| 4 | **Brecha de señal** | RedEdge3 tiene una brecha 4× mayor que SAR-VH |
| 5 | **Variabilidad** | RF es el más consistente entre experimentos (Std=0.008) |

> Estos hallazgos son observacionales — no asumen causalidad. El siguiente notebook los contextualiza con implicaciones para Colombia.